In [1]:
# Code for cross validation and splitting training data and test data
# IMPORTANT!!! The python version used for this code is 3.13.5, you need this exact version to get the same results 

In [1]:
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
import random

In [ ]:
df = pd.read_csv("../data/clean_data.csv")        
df = df.sample(frac=1, random_state=40).reset_index(drop=True)      # Shuffle data

In [4]:
# IMPORTANT!!!! To check that the code is reproducible, you should get the exact order of images shown below (after shuffliing)
df.head(10)

,Unnamed: 0,patient_id,lesion_id,smoke,drink,background_father,background_mother,age,pesticide,gender,...,itch,grew,hurt,changed,bleed,elevation,img_id,biopsed,group_id,2_3_lesions
0,1944,PAT_161,250,False,False,GERMANY,GERMANY,86,False,FEMALE,...,True,False,False,False,False,False,PAT_161_250_197.png,True,O,False
1,1979,PAT_573,1090,False,False,POMERANIA,POMERANIA,51,True,FEMALE,...,True,True,True,False,False,True,PAT_573_1090_164.png,True,J,False
2,1421,PAT_143,213,True,True,POMERANIA,POMERANIA,71,True,MALE,...,True,True,False,False,False,True,PAT_143_213_489.png,True,G,False
3,462,PAT_2071,4433,NaN,NaN,NaN,NaN,57,NaN,NaN,...,False,True,False,False,False,True,PAT_2071_4433_848.png,False,E,False
4,800,PAT_705,1326,False,True,GERMANY,GERMANY,58,True,FEMALE,...,True,True,False,False,False,True,PAT_705_1326_82.png,True,Q,False
5,511,PAT_1719,3203,NaN,NaN,NaN,NaN,83,NaN,NaN,...,True,False,False,False,False,True,PAT_1719_3203_18.png,False,B,False
6,458,PAT_1330,1172,NaN,NaN,NaN,NaN,24,NaN,NaN,...,False,False,False,False,False,True,PAT_1330_1172_722.png,False,P,False
7,1236,PAT_1186,681,NaN,NaN,NaN,NaN,38,NaN,NaN,...,False,False,True,False,False,False,PAT_1186_681_457.png,False,J,False
8,1920,PAT_1333,1177,NaN,NaN,NaN,NaN,38,NaN,NaN,...,False,True,False,False,False,True,PAT_1333_1177_434.png,False,J,True
9,1842,PAT_361,743,False,False,POMERANIA,POMERANIA,70,True,MALE,...,False,True,False,True,False,True,PAT_361_743_82.png,True,O,False


In [7]:
# The method below uses cross-validation method for splitting training and testing data (basically splitting 5 times)
# The method ensures that alomst 80% of EACH diagnosis is in the training data, and 20% in the test data 
# The method ensures that all images belonging to the same patient are in either the training group or test group


df['combined_y'] = df["diagnostic"].astype(str) + "_" + df["2_3_lesions"].astype(str)

# 2. Initialize and split
# We specify n_splits=5 explicitly for clarity and control
sgkf = StratifiedGroupKFold(n_splits=5)
sgkf_generator = sgkf.split(X=df, y=df['combined_y'], groups=df['patient_id'])

# 3. Populate the all_splits list
all_splits = [fold for fold in sgkf_generator]


# For each split_n, split_n[0] is for the training data, and split_n[0] is for the test data
# These only include the iindices for the data
split_0 = all_splits[0]
split_1 = all_splits[1]
split_2 = all_splits[2]
split_3 = all_splits[3]
split_4 = all_splits[4]

/opt/anaconda3/envs/pdsproject/lib/python3.12/site-packages/sklearn/model_selection/_split.py:1037: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(


In [8]:
df.loc[split_0[1], "group_id"] = "a"
df.loc[split_1[1], "group_id"] = "b"
df.loc[split_2[1], "group_id"] = "c"
df.loc[split_3[1], "group_id"] = "d"
df.loc[split_4[1], "group_id"] = "e"

In [9]:
df.to_csv("clean_data_with_splits.csv", index=False)

**THE REST BELOW IS JUST FOR CHECKING ACCURACY OF CODE**

In [10]:
# Checking that there is no data leakage
print( 
set(df.iloc[split_0[0]]["patient_id"]).isdisjoint(set(df.iloc[split_0[1]]["patient_id"])), 
set(df.iloc[split_1[0]]["patient_id"]).isdisjoint(set(df.iloc[split_1[1]]["patient_id"])), 
set(df.iloc[split_2[0]]["patient_id"]).isdisjoint(set(df.iloc[split_2[1]]["patient_id"])), 
set(df.iloc[split_3[0]]["patient_id"]).isdisjoint(set(df.iloc[split_3[1]]["patient_id"])), 
set(df.iloc[split_4[0]]["patient_id"]).isdisjoint(set(df.iloc[split_4[1]]["patient_id"])) 
)

True True True True True


In [11]:
# Checking stratification of data
for disease in df["diagnostic"].unique():
    data0 = df.iloc[split_0[0]]         # simply change [0] to [1] for checking the test data
    data1 = df.iloc[split_1[0]]
    data2 = df.iloc[split_2[0]]
    data3 = df.iloc[split_3[0]]
    data4 = df.iloc[split_4[0]]
    print(f"{disease}, {round(len(df[df["diagnostic"] == disease])/len(df), 3)}")
    
    print(round(len(data0[data0["diagnostic"] == disease])/len(data0), 3))
    print(round(len(data1[data1["diagnostic"] == disease])/len(data1), 3))
    print(round(len(data2[data2["diagnostic"] == disease])/len(data2), 3))
    print(round(len(data3[data3["diagnostic"] == disease])/len(data3), 3))
    print(round(len(data4[data4["diagnostic"] == disease])/len(data4), 3))
    print("\n")

BCC, 0.4
0.4
0.401
0.4
0.4
0.399


ACK, 0.268
0.269
0.268
0.267
0.267
0.269


SCC, 0.094
0.095
0.094
0.095
0.095
0.094


SEK, 0.096
0.096
0.097
0.096
0.096
0.096


NEV, 0.116
0.116
0.115
0.116
0.116
0.116


MEL, 0.026
0.025
0.025
0.026
0.026
0.026




In [12]:
# Checking that 80% of each diagnostic is present in the training data 
for disease in df["diagnostic"].unique():
    data0 = df.iloc[split_0[0]]         # simply change [0] to [1] for checking the test data
    data1 = df.iloc[split_1[0]]
    data2 = df.iloc[split_2[0]]
    data3 = df.iloc[split_3[0]]
    data4 = df.iloc[split_4[0]]
    print(disease)
    print(round(len(data0[data0["diagnostic"] == disease])/len(df[df["diagnostic"] == disease]), 3))
    print(round(len(data1[data1["diagnostic"] == disease])/len(df[df["diagnostic"] == disease]), 3))
    print(round(len(data2[data2["diagnostic"] == disease])/len(df[df["diagnostic"] == disease]), 3))
    print(round(len(data3[data3["diagnostic"] == disease])/len(df[df["diagnostic"] == disease]), 3))
    print(round(len(data4[data4["diagnostic"] == disease])/len(df[df["diagnostic"] == disease]), 3))
    print("\n")

BCC
0.799
0.802
0.8
0.8
0.799


ACK
0.802
0.8
0.798
0.798
0.802


SCC
0.801
0.795
0.801
0.807
0.795


SEK
0.799
0.804
0.799
0.799
0.799


NEV
0.801
0.796
0.801
0.801
0.801


MEL
0.771
0.792
0.812
0.812
0.812


